<a href="https://colab.research.google.com/github/azizegumus/ecommerce-customer-segmentation-rfm/blob/main/rfm_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Veriyi oku ve ilk 5 satırını ekrana getir
df = pd.read_csv('data.csv.csv', encoding='unicode_escape')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
print("veri boyutu:",df.shape) #df.shape kaç satır kaç sütun oldugunu gösteriyor

df.isnull().sum() # Hangi sütunda kaç tane boş (NaN) değer var?

veri boyutu: (541909, 8)


,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [4]:
df= df.dropna(subset=['CustomerID']) #dece CustomerID sütununa bakar, orası boşsa o satırı tamamen tablodan atar.

print("temizlik sonrası boyut:", df.shape)

temizlik sonrası boyut: (406829, 8)


In [5]:
# InvoiceNo sütununu metin (string) yapıp "C" ile başlamayanları ve Quantity değeri 0'dan büyük olanları tutuyoruz
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]

# Güncel boyutu görelim
print("İptaller Çıktıktan Sonra Boyut:", df.shape)

İptaller Çıktıktan Sonra Boyut: (397924, 8)


In [6]:
# Her satırın toplam harcamasını hesaplayıp yeni bir sütun olarak ekliyoruz
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# İlk 3 satıra bakıp sütunu kontrol edelim
df[['Quantity', 'UnitPrice', 'TotalPrice']].head(3)

,Quantity,UnitPrice,TotalPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00


In [7]:
# InvoiceDate sütununu tarih/saat formatına dönüştürüyoruz
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Veri setindeki en son alışveriş tarihini görelim
print("En son sipariş tarihi:", df['InvoiceDate'].max())

En son sipariş tarihi: 2011-12-09 12:50:00


In [11]:
import datetime as dt

# 1. En son sipariş tarihinden 1 gün sonrasını referans gün olarak alıyoruz
today_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
print("Analiz referans tarihi:", today_date)

# 2. Müşteri bazında Recency, Frequency ve Monetary hesaplıyoruz
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda date: (today_date - date.max()).days,
    'InvoiceNo': lambda num: num.nunique(),
    'TotalPrice': lambda price: price.sum()
})

# 3. Sütun isimlerini veriyoruz
rfm.columns = ['Recency', 'Frequency', 'Monetary']

# İlk 5 müşteriyi görelim
rfm.head()

Analiz referans tarihi: 2011-12-10 12:50:00


,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


In [12]:
# 1. Recency Skoru (Az gün = 5)
rfm["recency_score"] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

# 2. Monetary Skoru (Çok harcama = 5)
rfm["monetary_score"] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

# 3. Frequency Skoru (Çok sipariş = 5)
rfm["frequency_score"] = pd.qcut(rfm['Frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

# 4. İki haneli RFM Skoru (Örn: "55", "12")
rfm["RFM_SCORE"] = (rfm['recency_score'].astype(str) + rfm['frequency_score'].astype(str))
rfm.head()

,Recency,Frequency,Monetary,recency_score,monetary_score,frequency_score,RFM_SCORE
CustomerID,,,,,,,
12346.0,326,1,77183.60,1,5,1,11
12347.0,2,7,4310.00,5,5,5,55
12348.0,75,4,1797.24,2,4,4,24
12349.0,19,1,1757.55,4,4,1,41
12350.0,310,1,334.40,1,2,1,11


In [16]:
# Segment eşleştirme haritası (Regex kuralları)
seg_map = {
    r'[1-2][1-2]': 'hibernating',
    r'[1-2][3-4]': 'at_Risk',
    r'[1-2]5': 'cant_loose',
    r'3[1-2]': 'about_to_sleep',
    r'33': 'need_attention',
    r'[3-4][4-5]': 'loyal_customers',
    r'41': 'promising',
    r'51': 'new_customers',
    r'[4-5][2-3]': 'potential_loyalists',
    r'5[4-5]': 'champions'
}

# Skorları segment isimleriyle değiştiriyoruz
rfm['segment'] = rfm['RFM_SCORE'].replace(seg_map, regex=True)
rfm.head()

,Recency,Frequency,Monetary,recency_score,monetary_score,frequency_score,RFM_SCORE,segment
CustomerID,,,,,,,,
12346.0,326,1,77183.60,1,5,1,11,hibernating
12347.0,2,7,4310.00,5,5,5,55,champions
12348.0,75,4,1797.24,2,4,4,24,at_Risk
12349.0,19,1,1757.55,4,4,1,41,promising
12350.0,310,1,334.40,1,2,1,11,hibernating


In [17]:
# Segment bazında müşteri sayısı ve metrik ortalamaları
rfm_summary = rfm.groupby('segment').agg({
    'Recency': ['mean', 'count'],
    'Frequency': ['mean'],
    'Monetary': ['mean']
})
rfm_summary

Recency        Frequency     Monetary
                           mean count       mean         mean
segment                                                      
about_to_sleep        53.504274   351   1.162393   461.061510
at_Risk              155.062069   580   2.865517  1076.506433
cant_loose           132.428571    63   8.380952  2796.155873
champions              5.876777   633  12.417062  6857.935482
hibernating          217.897653  1065   1.101408   487.707579
loyal_customers       33.469166   827   6.458283  2856.720328
need_attention        53.064516   186   2.327957   889.226398
new_customers          6.857143    42   1.000000   388.212857
potential_loyalists   17.123984   492   2.010163  1034.905467
promising             23.350000   100   1.000000   351.797800